## EDA: Factibilidad y objetivos (P2)

Se justifica si cada familia de técnicas es factible con los datos y qué se espera hallar.
Los valores ‹…› se completan con los resultados de `a_entendimiento` y `b_exploracion`

---
### 1. Factibilidad

#### a. Clustering y anomalías (Parte II)

- ‹n_p› proveedores y ‹n_e› entidades con features agregables (volumen, monto, competencia, concentración, rubros, éxito)
- KMeans++ escala sobre todos los proveedores (Spark MLlib); DBSCAN/HDBSCAN y jerárquico son O(n²): sobre **entidades
  completas** (viable en local) y una **muestra estratificada** de proveedores
- Montos con cola pesada (P1: mediana S/186k vs p99 S/1,135M) → log1p antes de estandarizar
- **Veredicto:** factible

#### b. Recomendación (Parte III)

- Matriz entidad × proveedor con densidad ‹d› y entidad × familia UNSPSC (cobertura ‹cobertura›)
- Señal implícita: n° de adjudicaciones y log1p(monto); sin ratings explícitos → ALS implícito
- Contenido: `descripcion`/`items_desc` (los `tokens` del P1) por proveedor → TF-IDF + ANN (IVF/HNSW del P1)
- Evaluación con hold-out **temporal** (2023-24 → 2025)
- **Veredicto:** factible

#### c. PageRank y HITS (Parte IV)

- Bipartito dirigido entidad → proveedor; grado con cola pesada (b_exploracion §1) → la centralidad discrimina
- HITS: hubs = entidades que compran a proveedores autoridad; authorities = proveedores que reciben de buenos hubs
- **Veredicto:** factible; PageRank con GraphFrames sobre el grafo completo, HITS en local sobre un subgrafo

#### d. Comunidades (Parte V)

- Grafo de co-participación con ‹aristas› aristas tras tope y umbral; Girvan-Newman / espectral exacto solo en un
  subgrafo (una región o familia, ≤ ~2k nodos)
- ‹pct_multi_segmento›% de proveedores en varios segmentos y consorcios presentes → AGM/BigCLAM para el solapamiento
- **Veredicto:** factible con submuestra para métodos exactos

#### e. Representación (Parte VI)

- SVD truncada de la matriz entidad × proveedor (embedder directo) y Node2Vec sobre co-participación
- Tarea: visualización t-SNE/UMAP contrastada con clústeres y comunidades; predicción de enlaces con split temporal
- **Veredicto:** factible

### 2. Objetivos

| Técnica | Pregunta | Artefacto (KDD) |
|---|---|---|
| Clustering / anomalías | PQ1, PQ2 | `features_proveedor`, `features_entidad` |
| Recomendación | PQ3 | `matriz_entidad_proveedor`, `matriz_entidad_familia` |
| PageRank / HITS | PQ1 | `grafo_bipartito` |
| Comunidades | PQ2 | `grafo_coparticipacion` |
| Embeddings | PQ2, PQ3 | matriz + `grafo_coparticipacion` |

### 3. Relación con el P1

- La competencia del P1 (postor único, contratación directa) entra como **feature** (`pct_postor_unico`, `pct_directa`)
- MinHash/LSH del P1 se reutiliza en la resolución de entidades